# Lesson 21 — Spatial Branch-and-Bound

## Learning objectives

1. State the spatial-B&B algorithm for nonconvex MINLP.
2. Recognize the role of FBBT, OBBT, and convex relaxations inside the tree.
3. Read a `discopt` global-mode log: bounds, gap, node count.
4. Diagnose when spatial B&B is failing (gap not closing).

## 1. Algorithm

```
Stack = {root box B}
LB = -inf, UB = +inf, incumbent = None
while gap > tol and Stack non-empty:
    pick node N (with box B_N)
    LB_N = solve convex relaxation over B_N
    if LB_N >= UB:                  prune by bound
    if relaxation feasible & integer-feasible:
        # try local NLP solve to get a *primal* point
        x_local = local_NLP(N)
        if x_local feasible: UB <- min(UB, f(x_local))
    if (UB - LB_N)/UB < tol: prune (LB sufficient)
    else:
        pick branching variable j (most violated, longest edge, etc.)
        split B_N along j -> two children, push
update LB <- min over open nodes
```

This is the spatial B&B of {cite:p}`Smith1999,Tawarmalani2005,Belotti2009`. The key difference from MILP B&B (Lesson 6) is **branching on continuous variables** to refine the convex relaxation.

## 2. Convex relaxation at a node

For each nonconvex term, build the McCormick (Lesson 16) or αBB underestimator over the *current* box. The relaxation is an LP (if all relaxations are linear) or convex NLP, solved cheaply at each node.

This is why **bound tightening** (Lesson 15) is so critical in MINLP — each tightening *retroactively* tightens every relaxation in the tree.

## 3. Branching

Continuous branching variables are picked by:

- **Most violated** — branch on the variable whose envelope inequality is most violated by the LP solution.
- **Longest edge** — bisect the longest box edge.
- **Pseudocost-style** (recent literature).

Where to split (golden ratio? mid-point?): typically at the LP solution value or midpoint, with safeguards.

## 4. Convergence and termination

Spatial B&B is **finite for ε-optimality** but not finite in exact arithmetic for general nonconvex MINLPs. Convergence is by gap squeezing: every infinite branching sequence must shrink some box dimension to zero, eventually making the relaxation exact at the limit point {cite:p}`HorstTuy1996,Floudas1999`.

Practical termination: gap < ε (relative or absolute), or node/time limit.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model, sin, cos

# Nonconvex MINLP: pump network design (toy)
m = Model("pump_net")
x = m.add_variable(lb=0, ub=10, name="x")     # flow
y = m.add_variable(lb=0, ub=10, name="y")     # pressure
m.add_constraint(x * y >= 6)                  # work
m.add_constraint(0.5 * x**2 + y <= 12)        # head loss
m.minimize(0.1 * x**3 + y)                    # power cost

r = m.solve(mode="global", verbose=True)
print(f"global: f* = {r.objective:.6f} at x = {r.x}")
print(f"closed gap to: {r.relative_gap:.2e}, nodes = {r.nodes}")


## 5. When spatial B&B fails

- **Loose bounds** → loose envelopes → loose relaxations → no pruning. Fix: bound tightening, OBBT, branching on bounds.
- **Symmetry** → many equivalent nodes. Fix: lex breaking (Lesson 29).
- **High dimension** → curse of dimensionality on box partitioning. Fix: structure (Benders, Lagrangian) when applicable.

## References
{cite:p}`Smith1999,Tawarmalani2005,Belotti2009,Belotti2013,Floudas1999,HorstTuy1996`.